# Example 12.2.1 / 1: Propagate Arrival Times with Time Windows

Companion notebook for **12.2.1 Routing Model Variants** in the AMPL lab Documentation, section "Propagate Arrival Times with Time Windows". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Propagate Arrival Times with Time Windows

A **time window** $[e_i, l_i]$ allows service at node $i$ to start between $e_i$ and $l_i$. The
arrival time at $j$ depends on the predecessor only if arc $(i,j)$ is used, so the propagation is
an implication: `Use[i,j] = 1 ==> Arrive[j] >= Arrive[i] + service[i] + cost[i,j]`. The `==>`
operator is explained on Implication and Indicator Constraints (AMPL lab documentation, section 7.5). Prefer it, or a tight
derived constant, to an arbitrary big-M. For arc $(i,j)$ a valid big-M form is

$$ Arrive_j \ge Arrive_i + s_i + t_{ij} - M_{ij}(1 - Use_{ij}), \quad M_{ij} = l_i + s_i + t_{ij} - e_j. $$

With `latest[c1]` 50, `service[c1]` 1, `travel[c1,c3]` 8 and `earliest[c3]` 15, the constant for
arc (c1,c3) is 50 + 1 + 8 - 15 = 44. Check units, and whether travel times depend on departure
time. Waiting is allowed because the propagation is a lower bound; equality would forbid it.

The model uses the five nodes of Routing Models (AMPL lab documentation, section 12.2), with travel time equal to travel
cost, service time 1 at each customer, departure from the depot at time 0, and an earliest arrival
of 15 at `c3`. `TravelTime` skips arcs into the depot, so the return needs no depot arrival time.
Two switches reproduce two common failures: `forbid_wait` adds the reverse inequality, which makes
propagation an equality, and `single_depot_time` also propagates into the depot. The run rebuilds
the route from `Use` and simulates the times instead of trusting `Arrive`, which is only a lower
bound:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set NODES ordered;                      # depot first, then customers
set ARCS within {NODES, NODES};         # allowed directed arcs
param cost {ARCS} >= 0;                 # travel time equals travel cost here
param service {NODES} >= 0;             # service duration at each node
param earliest {NODES} >= 0;            # time window opens
param latest {i in NODES} >= earliest[i];  # time window closes
param forbid_wait binary default 0;     # 1 = also forbid waiting (failure switch)
param single_depot_time binary default 0;  # 1 = also propagate into the depot (failure switch)

var Use {ARCS} binary;                  # 1 if the tour uses arc (i,j)
var Arrive {i in NODES} >= earliest[i], <= latest[i];  # lower bound on arrival time

minimize TravelCost: sum {(i,j) in ARCS} cost[i,j] * Use[i,j];  # total travel cost

subject to Leave {i in NODES}: sum {(i,j) in ARCS} Use[i,j] = 1;  # one departure from every node
subject to Enter {j in NODES}: sum {(i,j) in ARCS} Use[i,j] = 1;  # one arrival at every node
# indicator propagation; arcs into the depot skipped by default
subject to TravelTime {(i,j) in ARCS: j <> first(NODES) or single_depot_time = 1}:
    Use[i,j] = 1 ==> Arrive[j] >= Arrive[i] + service[i] + cost[i,j];
# reverse inequality: makes propagation an equality
subject to NoWaiting {(i,j) in ARCS: j <> first(NODES) and forbid_wait = 1}:
    Use[i,j] = 1 ==> Arrive[j] <= Arrive[i] + service[i] + cost[i,j];

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
nodes = ["depot", "c1", "c2", "c3", "c4"]
travel = {                      # row: from node, column: to node
    "depot": [0, 2, 2, 9, 9],
    "c1":    [2, 0, 1, 8, 9],
    "c2":    [2, 2, 0, 9, 8],
    "c3":    [9, 8, 9, 0, 1],
    "c4":    [9, 9, 7, 1, 0],
}
ampl.set["NODES"] = nodes
ampl.set["ARCS"] = [(i, j) for i in nodes for j in nodes if i != j]
ampl.param["cost"] = {(i, j): travel[i][k]
                      for i in nodes for k, j in enumerate(nodes) if i != j}
ampl.param["service"] = {"depot": 0, "c1": 1, "c2": 1, "c3": 1, "c4": 1}
ampl.param["earliest"] = {"depot": 0, "c1": 0, "c2": 0, "c3": 15, "c4": 0}
ampl.param["latest"] = {"depot": 0, "c1": 50, "c2": 50, "c3": 50, "c4": 50}

In [5]:
# run.py
def route_and_times():
    succ = {i: j for (i, j), used in ampl.var["Use"].to_dict().items() if used > 0.5}
    earliest = ampl.param["earliest"].to_dict()
    service = ampl.param["service"].to_dict()
    cost = ampl.param["cost"].to_dict()
    route, times, node, clock = ["depot"], {}, "depot", 0
    while True:
        nxt = succ[node]
        clock = max(clock + service[node] + cost[node, nxt], earliest[nxt])
        route.append(nxt)
        if nxt == "depot":
            return route, times, clock
        times[nxt], node = clock, nxt

for switch in ["none", "forbid_wait", "single_depot_time"]:
    ampl.param["forbid_wait"] = 1 if switch == "forbid_wait" else 0
    ampl.param["single_depot_time"] = 1 if switch == "single_depot_time" else 0
    ampl.solve(solver="highs", verbose=False)
    print(switch, ampl.solve_result)
    if ampl.solve_result == "solved":
        route, times, back = route_and_times()
        print(" cost", ampl.obj["TravelCost"].value(), "route", route)
        print(" simulated arrivals", times, "return", back)
        print(" every customer once:", sorted(route[1:-1]) == sorted(nodes[1:]))

none solved
 cost 20.0 route ['depot', 'c1', 'c3', 'c4', 'c2', 'depot']
 simulated arrivals {'c1': 2, 'c3': 15, 'c4': 17, 'c2': 25} return 28
 every customer once: True
forbid_wait solved
 cost 21.0 route ['depot', 'c1', 'c2', 'c4', 'c3', 'depot']
 simulated arrivals {'c1': 2, 'c2': 4, 'c4': 13, 'c3': 15} return 25
 every customer once: True
single_depot_time infeasible


### Pure AMPL: data.dat and model.run

Pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set NODES := depot c1 c2 c3 c4;
set ARCS:  depot c1 c2 c3 c4 :=
    depot    -    +  +  +  +
    c1       +    -  +  +  +
    c2       +    +  -  +  +
    c3       +    +  +  -  +
    c4       +    +  +  +  - ;
param cost:  depot c1 c2 c3 c4 :=
    depot    .    2  2  9  9
    c1       2    .  1  8  9
    c2       2    2  .  9  8
    c3       9    8  9  .  1
    c4       9    9  7  1  . ;
param: service earliest latest :=
    depot   0    0   0
    c1      1    0  50
    c2      1    0  50
    c3      1   15  50
    c4      1    0  50 ;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs, solver_msg 0;
for {switch in 0..2} {
    let forbid_wait := if switch = 1 then 1 else 0;
    let single_depot_time := if switch = 2 then 1 else 0;
    solve;
    printf "forbid_wait %d single_depot_time %d: %s", forbid_wait, single_depot_time, solve_result;
    if solve_result = "solved" then printf ", cost %g", TravelCost;
    printf "\n";
}

forbid_wait 0 single_depot_time 0: solved, cost 20
forbid_wait 1 single_depot_time 0: solved, cost 21

suffix dunbdd OUT;
forbid_wait 0 single_depot_time 1: infeasible
